# ⚙️ 02 - Data Preprocessing & Feature Engineering

Thực hiện tiền xử lý dữ liệu và trích xuất đặc trưng số học theo yêu cầu đề bài:
- **1. Làm sạch văn bản**: Loại bỏ stop words, punctuation, HTML tags/URLs, chuẩn hóa Unicode
- **2. Phân chia tập dữ liệu độc lập** (Early Stratified Split 80/10/10) chống rò rỉ dữ liệu
- **3. Vector hóa văn bản bằng TF-IDF** (TfidfVectorizerScratch - Unigram & Bigram)
- **4. Trích xuất đặc trưng hình thức** (Tần suất ký tự '!', '$', chữ in hoa, độ dài)
- **5. Ghép nối không gian ma trận lai** (Hybrid Feature Matrix: Word + Char + Keywords + Numerics)
- **6. Thử nghiệm các tổ hợp đặc trưng** (Feature Combinations Experiment)


Notebook này **không gọi trực tiếp các model có sẵn** như `sklearn.linear_model`,
`sklearn.svm`, `sklearn.naive_bayes`, `sklearn.ensemble`, `sklearn.model_selection`
hoặc `sklearn.feature_extraction`.

Các thuật toán được gọi từ package tự xây dựng của project:

```text
src/
├── data/
├── features/
├── models/
├── evaluation/
└── deployment/
```

Các thư viện tính toán nền tảng (NumPy/Pandas/Scipy/Matplotlib...) nếu có được sử dụng
thì được **ẩn bên trong `src/`**, còn notebook chỉ làm việc với API của project.

## 1. Import Thư Viện & Cấu Hình Môi Trường
- Thiết lập đường dẫn động `PROJECT_ROOT` từ `Path.cwd()`.
- Nạp các module tự xây dựng từ `src.data` và `src.features`.


In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Thiết lập đường dẫn gốc tương đối an toàn
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SpamExperimentConfig
from src.data.loader import load_raw_data
from src.data.preprocessing import (
    clean_email_split,
    stratified_split_dataframe,
    stratified_split_records,
    SpamTextProcessor,
    audit_data_quality,
)
from src.features.feature_engineering import (
    CSRMatrixScratch,
    TfidfVectorizerScratch,
    MaxAbsScalerScratch,
    SpamSignalFeatureExtractor,
    HybridFeatureBuilderScratch,
    experiment_feature_combinations,
    export_processed_data,
)

print("Project root:", PROJECT_ROOT)
print("Nạp các module Task 1 thành công!")


Project root: D:\dowload\spam_classifier_task2_completed\spam_classifier-main
Nạp các module Task 1 thành công!


## 2. Làm Sạch Toàn Bộ Dữ Liệu & Phân Tầng Độc Lập (Early Stratified Split)
- **Quy tắc chống rò rỉ dữ liệu (Data Leakage)**:
  1. Gọi `clean_email_split()` trên TOÀN BỘ tập dữ liệu thô để loại bỏ dòng rỗng, nhãn không hợp lệ, và xử lý mâu thuẫn nhãn/duplicate chéo.
  2. Phân tầng sớm 80% Train - 10% Validation - 10% Test bằng `stratified_split_records()` với `random_state=42`.


In [2]:
# 1. Nạp dữ liệu thô
raw_records = load_raw_data(PROJECT_ROOT / "data" / "raw" / "spam.csv")
print(f"Tổng số bản ghi thô: {len(raw_records):,}")

# 2. Kiểm tra chất lượng dữ liệu
quality_report = audit_data_quality(raw_records)
print("Báo cáo kiểm toán dữ liệu:", quality_report)

# 3. Làm sạch chất lượng toàn diện trước khi chia tập
clean_records = clean_email_split(raw_records)
print(f"Số bản ghi sạch giữ lại: {len(clean_records):,}")

# 4. Chia phân tầng 80/10/10
train_df, val_df, test_df = stratified_split_records(
    clean_records,
    label_col="label",
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    random_state=42,
)

# 5. Kiểm tra phân phối các tập con
for name, split_data in [("Train (80%)", train_df), ("Val (10%)", val_df), ("Test (10%)", test_df)]:
    counts = pd.Series([r["label"] for r in split_data]).value_counts()
    pcts = (counts / len(split_data) * 100).round(2)
    print(f"\n--- {name}: {len(split_data):,} mẫu ---")
    for lbl in ["ham", "spam"]:
        print(f"  {lbl.upper()}: {counts.get(lbl, 0):,} ({pcts.get(lbl, 0.0)}%)")


Tổng số bản ghi thô: 43,849
Báo cáo kiểm toán dữ liệu: {'input_rows': 43849, 'invalid_label_rows': 2195, 'empty_text_rows': 2192, 'conflicting_groups': 738, 'conflicting_rows': 1480, 'duplicate_rows_removed': 8954, 'retained_rows': 29028}
Số bản ghi sạch giữ lại: 29,028

--- Train (80%): 23,223 mẫu ---
  HAM: 13,309 (57.31%)
  SPAM: 9,914 (42.69%)

--- Val (10%): 2,903 mẫu ---
  HAM: 1,664 (57.32%)
  SPAM: 1,239 (42.68%)

--- Test (10%): 2,902 mẫu ---
  HAM: 1,663 (57.31%)
  SPAM: 1,239 (42.69%)


## 3. Tiền Xử Lý Văn Bản & Trích Xuất Đặc Trưng Cấu Trúc (SpamTextProcessor)
- Chuẩn hóa Unicode NFKD, lowercase.
- Xóa thẻ HTML bằng Parser tự viết.
- Token hóa URL (`urltoken`), số tiền (`moneytoken`), số (`numbertoken`).
- Loại bỏ stop words và dấu câu.
- Trích xuất đặc trưng cấu trúc: độ dài, chữ hoa, số lượng dấu `!`, `$`, `?`.


In [3]:
processor = SpamTextProcessor()

# Minh họa làm sạch trên 3 email mẫu
for i, email in enumerate(train_df[:3], 1):
    raw_text = email.get("raw_text", email.get("text"))
    cleaned = processor.normalize_text(raw_text)
    struct = processor.extract_structural_features(raw_text)
    print(f"\n[Mẫu {i}] Nhãn: {email.get('label').upper()}")
    print("Văn bản gốc:", raw_text[:120], "...")
    print("Văn bản sạch:", cleaned[:120], "...")
    print("Tín hiệu cấu trúc:", {k: struct[k] for k in ["uppercase_ratio", "exclamation_count", "dollar_count", "has_url"]})



[Mẫu 1] Nhãn: HAM
Văn bản gốc: Maybe you should find something else to do instead??? ...
Văn bản sạch: maybe find something else instead ...
Tín hiệu cấu trúc: {'uppercase_ratio': 0.023809523809523808, 'exclamation_count': 0.0, 'dollar_count': 0.0, 'has_url': 0.0}

[Mẫu 2] Nhãn: HAM
Văn bản gốc: How are you enjoying this semester? Take care brother. ...
Văn bản sạch: how enjoying semester take care brother ...
Tín hiệu cấu trúc: {'uppercase_ratio': 0.045454545454545456, 'exclamation_count': 0.0, 'dollar_count': 0.0, 'has_url': 0.0}

[Mẫu 3] Nhãn: SPAM
Văn bản gốc:  escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapen ...
Văn bản sạch: escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenumberd escapenu ...
Tín hiệu cấu trúc: {'uppercase_ratio': 0.0, 'exclamation_count': 0.0, 'dollar_count': 0.0, 'has_url': 0.0}


## 4. Vector Hóa Văn Bản Bằng TF-IDF From-Scratch (TfidfVectorizerScratch)
- Sublinear Term Frequency: $\text{TF} = 1 + \ln(\text{count})$.
- Smooth Inverse Document Frequency: $\text{IDF} = \ln((1 + N)/(1 + \text{DF})) + 1$.
- Chuẩn hóa vector $L_2$ norm.
- **Chỉ fit trên Train**, sau đó transform trên Val và Test.


In [4]:
# Chuẩn bị cột model_text
train_prepared = processor.prepare_feature_frame(train_df)
val_prepared = processor.prepare_feature_frame(val_df)
test_prepared = processor.prepare_feature_frame(test_df)

train_texts = [r["model_text"] for r in train_prepared]
val_texts = [r["model_text"] for r in val_prepared]

# Fit TF-IDF chỉ trên Train
tfidf = TfidfVectorizerScratch(min_df=2, max_features=3000, sublinear_tf=True, smooth_idf=True, norm="l2")
tfidf.fit(train_texts)

print(f"Số lượng từ vựng (Vocabulary size): {len(tfidf.vocabulary_):,}")

# Top từ có IDF cao nhất và thấp nhất
terms_by_idf = sorted(tfidf.vocabulary_.items(), key=lambda item: tfidf.idf_diag_[item[1]], reverse=True)
print("\nTop 10 từ có IDF cao nhất (từ hiếm/đặc trưng):", [t[0] for t in terms_by_idf[:10]])
print("Top 10 từ có IDF thấp nhất (từ phổ biến nhất):", [t[0] for t in terms_by_idf[-10:]])

X_train_word = tfidf.transform(train_texts)
X_val_word = tfidf.transform(val_texts)
print(f"Kích thước ma trận từ vựng Train: {X_train_word.shape}, Val: {X_val_word.shape}")


Số lượng từ vựng (Vocabulary size): 3,000

Top 10 từ có IDF cao nhất (từ hiếm/đặc trưng): ['affected', 'agenda', 'blog', 'closer', 'contacting', 'ended', 'ending', 'equivalent', 'escapenumberc', 'escapenumbert']
Top 10 từ có IDF thấp nhất (từ phổ biến nhất): ['time', 'get', 'e', 'one', 'numbertoken', 'not', 'please', 'com', 'http', 'escapenumber']
Kích thước ma trận từ vựng Train: (23223, 3000), Val: (2903, 3000)


## 5. Xây Dựng Ma Trận Đặc Trưng Lai Toàn Diện & Đối Sánh Các Tổ Hợp Đặc Trưng
- Ghép nối đa khối: `[Word TF-IDF + Bigrams + Char N-grams + Keywords + Scaled Numerics]`.
- Đối sánh 5 tổ hợp đặc trưng theo yêu cầu đề bài:
  1. `word_tfidf`: Chỉ dùng TF-IDF từ đơn.
  2. `word_char_frequency`: Word TF-IDF + tần suất ký tự đặc biệt (`!`, `$`).
  3. `word_char_ngrams`: Word TF-IDF + Character n-grams (3-5 ký tự).
  4. `word_keywords`: Word TF-IDF + 10 nhóm từ khóa spam có trọng số.
  5. `hybrid`: Kết hợp toàn bộ các nguồn đặc trưng.


In [ ]:
config = SpamExperimentConfig()
signal_extractor = SpamSignalFeatureExtractor(config)
builder = HybridFeatureBuilderScratch(config, processor, signal_extractor)

# Fit trên Train và Transform trên Val & Test
X_train_hybrid = builder.fit_transform(train_df)
X_val_hybrid = builder.transform(val_df)
X_test_hybrid = builder.transform(test_df)

print(f"Tổng số đặc trưng lai: {X_train_hybrid.shape[1]:,} chiều")
print("Các phân đoạn khối đặc trưng:", builder.feature_slices_)

# Thử nghiệm 5 tổ hợp đặc trưng
combinations = experiment_feature_combinations(train_df, val_df)
print("\n===== BẢNG ĐỐI SÁNH KÍCH THƯỚC CÁC TỔ HỢP ĐẶC TRƯNG =====")
summary_rows = []
for name, (xt, xv) in combinations.items():
    summary_rows.append({
        "Tổ hợp đặc trưng": name,
        "Số chiều (Features)": xt.shape[1],
        "Train Shape": str(xt.shape),
        "Val Shape": str(xv.shape),
        "NNZ (Train non-zeros)": xt.nnz,
        "Độ thưa (%)": round(100.0 * (1.0 - xt.nnz / (xt.shape[0] * xt.shape[1])), 4),
    })

df_combos = pd.DataFrame(summary_rows)
display(df_combos)


Tổng số đặc trưng lai: 4,318 chiều
Các phân đoạn khối đặc trưng: {'word': (0, 3000), 'bigram': (3000, 3800), 'char': (3800, 4300), 'keyword': (4300, 4310), 'numeric': (4310, 4318)}


## 6. Xuất Dữ Liệu Đệm (Cache Persistence) Vào data/processed/
- Lưu trữ các tập phân tầng sạch (`train_clean.csv`, `val_clean.csv`, `test_clean.csv`).
- Lưu trữ các ma trận đặc trưng CSR (`X_train.npz`, `X_val.npz`, `X_test.npz`) và vector nhãn `y`.
- Giúp Task 2 nạp ngay tức thì mà không phải lặp lại tiền xử lý tốn kém.


In [ ]:
export_result = export_processed_data(
    train_df, val_df, test_df, builder=builder
)
print("Kết quả kết xuất dữ liệu bộ đệm:", export_result)
print("Đã bàn giao đầy đủ đầu ra của Task 1 cho Task 2 thành công!")


Kết quả kết xuất dữ liệu bộ đệm: {'train_csv': 'E:\\Hoc_May_Tren_truong\\lab_1 spam_classifier\\data\\processed\\train_clean.csv', 'val_csv': 'E:\\Hoc_May_Tren_truong\\lab_1 spam_classifier\\data\\processed\\val_clean.csv', 'test_csv': 'E:\\Hoc_May_Tren_truong\\lab_1 spam_classifier\\data\\processed\\test_clean.csv', 'matrices_saved': True}
Đã bàn giao đầy đủ đầu ra của Task 1 cho Task 2 thành công!


In [1]:

from pathlib import Path
import joblib

# Thư mục lưu pipeline tiền xử lý của Task 1
save_dir = PROJECT_ROOT / "results" / "saved_models"
save_dir.mkdir(parents=True, exist_ok=True)

# Kiểm tra builder đã fit trên Train
assert builder.is_fitted_, "Builder chưa được fit trên tập Train!"

# Lưu processor và builder đã fit cùng nhau
feature_pipeline = {
    "text_processor": processor,
    "feature_builder": builder,
    "feature_names": builder.feature_names_,
    "feature_slices": builder.feature_slices_,
    "n_features": len(builder.feature_names_),
    "data_split": {
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df),
    },
    "random_state": 42,
}

artifact_path = save_dir / "task1_feature_pipeline.joblib"
joblib.dump(feature_pipeline, artifact_path)

print("Đã lưu pipeline Task 1:", artifact_path)
print("Builder fitted:", builder.is_fitted_)
print("Số chiều đặc trưng:", feature_pipeline["n_features"])
print("Số mẫu Train:", len(train_df))
print("Số mẫu Validation:", len(val_df))
print("Số mẫu Test:", len(test_df))


NameError: name 'PROJECT_ROOT' is not defined